# box-array-to-tensor-with-recipe — worked example 1: Box the output of a unary negate and attach its Recipe

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `box-array-to-tensor-with-recipe`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

When a forward op finishes, its raw array must be **boxed** back into a `MiniTensor` so the rest of the graph speaks one type. The boxed tensor inherits `requires_grad` from the gate the wrapper already computed. A `Recipe(func, args, kwargs, parents)` is attached **only when** `requires_grad` is True — it is the breadcrumb the reverse pass follows, and `recipe is None` is the leaf signal that stops traversal.

## Worked solution

We wrap a single unary op, `negate`, on one grad-tracked input.

1. **Unbox the input.** The raw numpy/torch fn cannot consume a `MiniTensor`, so we pull out `.array`. Here `raw_args = (x.array,)`.
2. **Run the raw forward.** `out_raw = negate(x.array)` gives a plain array — no graph metadata yet.
3. **Decide the gate.** With one rg=True input and grad tracking on, `requires_grad = True`.
4. **Box.** Wrap the raw array: `out = MiniTensor(out_raw, requires_grad=True)`. Boxing is unconditional — callers always expect a `MiniTensor`, even in no_grad.
5. **Attach the Recipe.** Because `requires_grad` is True, we record `Recipe(negate, raw_args, {}, {0: x})`. The `parents` dict maps argnum 0 to the input tensor so the backward pass can route the gradient. Had the gate been False, we would leave `out.recipe = None` and skip this entirely.

In [ ]:
from dataclasses import dataclass, field
from typing import Callable

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = np.asarray(array)
        self.requires_grad = requires_grad
        self.recipe = None

@dataclass
class Recipe:
    func: Callable
    args: tuple
    kwargs: dict
    parents: dict

def negate(a):
    return -a

def box_negate(x: MiniTensor) -> MiniTensor:
    raw_args = (x.array,)
    requires_grad = x.requires_grad
    out_raw = negate(*raw_args)
    out = MiniTensor(out_raw, requires_grad=requires_grad)
    if requires_grad:
        out.recipe = Recipe(negate, raw_args, {}, {0: x})
    return out

x = MiniTensor(np.array([1.0, -2.0, 3.0]), requires_grad=True)
out = box_negate(x)
print("array      :", out.array)
print("requires_grad:", out.requires_grad)
print("recipe func:", out.recipe.func.__name__)
print("recipe parents keys:", list(out.recipe.parents.keys()))
print("parent[0] is x:", out.recipe.parents[0] is x)